<a href="https://colab.research.google.com/github/apeksha300/Computer_vision_/blob/main/Computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

import os
import cv2
import numpy as np
from ultralytics import YOLO

input_video = "/content/demo_videoo.mp4"
output_video = "/content/pose_CV.mp4"

if not os.path.exists(input_video):
    raise FileNotFoundError(f"File '{input_video}' not found. Please upload it to the Colab files section.")

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise ValueError("Could not open video file. Check file path or format.")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

out = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

model = YOLO("yolov8n-pose.pt")

SKELETON_CONNECTIONS = [
    (16, 14), (14, 12), (17, 15), (15, 13), (12, 13),
    (6, 12), (5, 11), (6, 5),
    (6, 8), (8, 10), (5, 7), (7, 9),
    (3, 1), (1, 2), (2, 4), (1, 0), (2, 0)
]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    black_frame = np.zeros((height, width, 3), dtype=np.uint8)

    results = model(frame, verbose=False)

    if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
        for person in results[0].keypoints.xy:
            keypoints = person.cpu().numpy()
            
            for p1_idx, p2_idx in SKELETON_CONNECTIONS:
                if p1_idx < len(keypoints) and p2_idx < len(keypoints):
                    p1 = keypoints[p1_idx]
                    p2 = keypoints[p2_idx]
                    
                    if p1[0] > 0 and p1[1] > 0 and p2[0] > 0 and p2[1] > 0:
                        pt1 = (int(p1[0]), int(p1[1]))
                        pt2 = (int(p2[0]), int(p2[1]))
                        
                        cv2.line(black_frame, pt1, pt2, (255, 255, 255), 2)

            for kp in keypoints:
                if kp[0] > 0 and kp[1] > 0:
                    cv2.circle(black_frame, (int(kp[0]), int(kp[1])), 3, (255, 255, 255), -1)

    out.write(black_frame)

cap.release()
out.release()

print("Process completed! Output video created at '/content/pose_CV.mp4'")